<a href="https://colab.research.google.com/github/ArinaKrasotina/data-analysis/blob/main/%D0%9A%D1%80%D0%B0%D1%81%D0%BE%D1%82%D0%B8%D0%BD%D0%B0_%22%D0%90%D0%BA%D1%82%D0%B8%D0%B2%D0%BD%D0%BE%D1%81%D1%82%D1%8C_Itresume%22.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## **2 кейс**

**Выгрузка активности с ItResume**

**Важно**

Перед началом решения выполните следующую ячейку, чтобы загрузить необходимый для работы файл.

In [ ]:
!wget https://gist.github.com/Vs8th/a7a7f00e6cdef1b3fe87e4d61ca56e5f/raw/codesubmit.csv

--2025-12-10 10:47:20--  https://gist.github.com/Vs8th/a7a7f00e6cdef1b3fe87e4d61ca56e5f/raw/codesubmit.csv
Resolving gist.github.com (gist.github.com)... 140.82.113.3
Connecting to gist.github.com (gist.github.com)|140.82.113.3|:443... connected.
HTTP request sent, awaiting response... 301 Moved Permanently
Location: https://gist.githubusercontent.com/Vs8th/a7a7f00e6cdef1b3fe87e4d61ca56e5f/raw/codesubmit.csv [following]
--2025-12-10 10:47:20--  https://gist.githubusercontent.com/Vs8th/a7a7f00e6cdef1b3fe87e4d61ca56e5f/raw/codesubmit.csv
Resolving gist.githubusercontent.com (gist.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to gist.githubusercontent.com (gist.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 215378 (210K) [text/plain]
Saving to: ‘codesubmit.csv’

codesubmit.csv      100%[===================>] 210.33K  --.-KB/s    in 0.03s   

2025-12-10 10:47:21 (8.09 MB/s) - ‘co

Чтобы посмотреть как он выглядит выполните следующую ячейку.

In [ ]:
import pandas as pd

df = pd.read_csv('codesubmit.csv', sep = ';')
df

,created_at,user_id,problem_id,is_correct,type
0,2023-04-30 13:47:14.344471,7,870,1.0,submit
1,2023-04-30 13:46:15.949925,7,870,0.0,submit
2,2023-04-30 16:13:26.005286,173,21,1.0,submit
3,2023-04-30 16:13:06.739782,173,21,NaN,run
4,2023-04-30 15:52:00.195532,173,25,1.0,submit
...,...,...,...,...,...
4994,2023-04-30 21:52:00.269123,13493,435,NaN,run
4995,2023-04-30 21:51:01.094234,13493,435,1.0,submit
4996,2023-04-30 21:50:52.059690,13493,435,NaN,run
4997,2023-04-30 21:42:24.323689,13493,1086,NaN,run


### **Решения**

#### **Задача 1**

Ваша задача - выяснить сколько в среднем тратится времени на решение задачи.

**Примечание**: для правильного подсчета - рассчитайте сначала среднее время решения по каждой задаче в отдельности, и только затем находите общее среднее время решения задач.

Результат - число типа `float`, округлите до 2 знаков после запятой и запишите в переменную `res`.


**Решение**

Напишите свое решение ниже

In [98]:
import csv
from datetime import datetime
from collections import defaultdict
from statistics import mean

data = []
with open('codesubmit.csv') as f:
    reader = csv.DictReader(f, delimiter=';')
    for row in reader:
        row['created_at'] = datetime.strptime(row['created_at'], '%Y-%m-%d %H:%M:%S.%f')
        row['is_correct'] = float(row['is_correct']) if row['is_correct'] else None
        data.append(row)

data.sort(key=lambda x: x['created_at'])

grouped = defaultdict(list)
for row in data:
    key = (row['problem_id'], row['user_id'])
    grouped[key].append(row)

problem_times = defaultdict(list)

for (problem_id, user_id), attempts in grouped.items():
    first_correct_submit = None
    for attempt in attempts:
        if attempt['type'] == 'submit' and attempt['is_correct'] == 1:
            first_correct_submit = attempt
            break

    if not first_correct_submit:
        continue

    first_activity = attempts[0]

    if first_activity == first_correct_submit:
        continue

    time_diff = (first_correct_submit['created_at'] - first_activity['created_at']).total_seconds()
    problem_times[problem_id].append(time_diff)

problem_averages = {}
for problem_id, times in problem_times.items():
    if times:
        problem_averages[problem_id] = mean(times)

if problem_averages:
    res = round(mean(problem_averages.values()), 2)
else:
    res = 0

print(f"Результат: {res}")

Результат: 611.86


✏️ ✏️ ✏️

**Проверка**

Чтобы проверить свое решение, запустите код в следующих ячейках

In [99]:
try:
    assert res == 611.86
except AssertionError:
    print('Ответы не совпадают')
else:
    print('Поздравляем, Вы справились!')

Поздравляем, Вы справились!


#### **Задача 2**

Ваша задача - выяснить сколько часов в среднем проводит юзер в день на платформе. Перерывы в активности за день - не учитываем.

Результат - число типа `float`, округлите до 2 знаков после запятой и запишите в переменную `res2`.

**Решение**

Напишите свое решение ниже

In [ ]:
df['created_at'] = pd.to_datetime(df['created_at'])
df['date'] = df['created_at'].dt.date

def active_time(group):
    g = group.sort_values('created_at')
    return (g['created_at'].iloc[-1] - g['created_at'].iloc[0]).total_seconds() / 3600

# считаем для каждого user–date
daily = df.groupby(['user_id','date']).apply(active_time)

res2 = round(daily.mean(), 2)
res2


/tmp/ipython-input-3014883123.py:9: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  daily = df.groupby(['user_id','date']).apply(active_time)


np.float64(1.7)

✏️ ✏️ ✏️

**Проверка**

Чтобы проверить свое решение, запустите код в следующих ячейках

In [ ]:
try:
    assert res2 == 1.7
except AssertionError:
    print('Ответы не совпадают')
else:
    print('Поздравляем, Вы справились!')

Поздравляем, Вы справились!


#### **Задача 3**

Теперь давайте посмотрим на активные сеансы. Выясните, сколько задач в среднем решается за один активный сеанс.

**Активный сеанс** - период, когда между любой активностью пользователя разница менее или равна часу, не более

**Важно**: в расчет берем не только успешные попытки решений (`is_correct=1`), а и неуспешные тоже (`is_correct=0`), и тип `run` в том числе.

Результат - число типа `float`, округлите до 2 знаков после запятой и запишите в переменную `res3`.

**Решение**

Напишите свое решение ниже

In [ ]:
df['created_at'] = pd.to_datetime(df['created_at'])
df = df.sort_values(['user_id', 'created_at'])

sessions = []
for user, g in df.groupby('user_id'):
    g = g.sort_values('created_at')
    diff = g['created_at'].diff().dt.total_seconds().fillna(0)
    session_id = (diff > 3600).cumsum()   # новый сеанс
    g = g.assign(session=session_id)

    sessions.append(g)

sessions = pd.concat(sessions)

# считаем число уникальных задач в каждом сеансе
session_tasks = (
    sessions
    .groupby(['user_id', 'session'])['problem_id']
    .nunique()
)

res3 = round(session_tasks.mean(), 2)
res3


np.float64(3.14)

✏️ ✏️ ✏️

**Проверка**

Чтобы проверить свое решение, запустите код в следующих ячейках

In [ ]:
try:
    assert res3 == 3.14
except AssertionError:
    print('Ответы не совпадают')
else:
    print('Поздравляем, Вы справились!')

Поздравляем, Вы справились!


#### **Задача 4**

И финальная - найдите самый "популярный" час дня на нашей платформе.

Популярность определяем максимальным количеством уникальных пользователей, совершающих какую-либо активность в этот период

Результат в числовом формате запишите в переменную `res4`.

Например, самым популярным часом стал период с 22 до 23, тогда в переменной `res4` должно лежать **22**. Обозначающее начало этого периода.

**Решение**

Напишите свое решение ниже

In [ ]:
df['hour'] = df['created_at'].dt.hour

popular = (
    df.groupby('hour')['user_id']
    .nunique()
    .sort_values(ascending=False)
)

res4 = int(popular.index[0])
res4


16

✏️ ✏️ ✏️

**Проверка**

Чтобы проверить свое решение, запустите код в следующих ячейках

In [ ]:
try:
    assert res4 == 16
except AssertionError:
    print('Ответы не совпадают')
else:
    print('Поздравляем, Вы справились!')

Поздравляем, Вы справились!
